In [2]:
import os, subprocess, shutil
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ.get("PATH", "")
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"
print("winutils:", os.path.exists(r"C:\hadoop\bin\winutils.exe"))
print("hadoop.dll:", os.path.exists(r"C:\hadoop\bin\hadoop.dll"))

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot" 
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + ";" + os.environ.get("PATH", "")
print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("java in PATH:", shutil.which("java"))
print(subprocess.run(["java","-version"], capture_output=True, text=True).stderr)

winutils: True
hadoop.dll: True
JAVA_HOME: C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot
java in PATH: C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot\bin\java.EXE
openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment Temurin-17.0.18+8 (build 17.0.18+8)
OpenJDK 64-Bit Server VM Temurin-17.0.18+8 (build 17.0.18+8, mixed mode, sharing)



In [3]:
import sys
# Affiche le chemin de l'exécutable Python utilisé par ce notebook
print(sys.executable)

c:\Users\etien\miniconda3\envs\nf26\python.exe


In [4]:
import os
from pyspark.sql import SparkSession

# Arrêter la session Spark existante pour appliquer les nouveaux paramètres
if 'spark' in locals():
    spark.stop()

# Définir explicitement le chemin de l'exécutable Python pour Spark
# Ce chemin correspond à l'environnement du kernel du notebook
python_path = r'c:\Users\etien\miniconda3\envs\nf26\python.exe'
os.environ['PYSPARK_PYTHON'] = python_path
os.environ['PYSPARK_DRIVER_PYTHON'] = python_path

# Recréer la SparkSession avec la configuration mémoire et la configuration du worker
spark = (
    SparkSession.builder
    .appName("GHG-Inventory-ETL")
    .config("spark.driver.memory", "8g")
    .config("spark.python.worker.exec", python_path)
    .config("spark.sql.ansi.enabled", "false")
    .getOrCreate()
)

import pyspark.pandas as ps
ps.set_option("compute.fail_on_ansi_mode", False)

print(f"PYSPARK_PYTHON est maintenant défini sur : {os.environ.get('PYSPARK_PYTHON')}")
print("Nouvelle session Spark créée avec succès.")

PYSPARK_PYTHON est maintenant défini sur : c:\Users\etien\miniconda3\envs\nf26\python.exe
Nouvelle session Spark créée avec succès.


In [5]:
import pandas as pd
import numpy as np
from pathlib import Path
import pyspark.pandas as ps
from pyspark.sql.functions import *
from datetime import date, timedelta

#### Fonction de chargement des données vers un spark pandas dataframe

In [6]:
def readFile(path, indexCol):
    return ps.read_csv(path, sep=';', index_col=indexCol)

#### Fonctions d'homogénéisation de la langue des fonctions et missions

In [7]:
def clean_langue_fonction(df, site):
    match site:
        case "BERLIN":
            map_fonctions = {
                "Ökonom": "Economist",
                "Führungskraft": "Business Executive",
                "Personalleiter": "HRD",
                "Computeringenieur": "Computer Engineer",
                "Dateningenieur": "Data Engineer",
            }
            df["FONCTION_PERSONNEL"] = df["FONCTION_PERSONNEL"].replace(map_fonctions)
        case "PARIS":
            map_fonctions = {
                "Ingénieur Informaticien": "Computer Engineer",
                "Ingénieur Data": "Data Engineer",
                "Economiste": "Economist",
                "DRH": "HRD",
                "Cadre": "Business Executive",
            }
            df["FONCTION_PERSONNEL"] = df["FONCTION_PERSONNEL"].replace(map_fonctions)


In [8]:
def clean_langue_mission(df, site):
    match site:
        case "BERLIN":
            map_type_mission = {
                "Geschäftstreffen": "Business Meeting",
                "Konferenz": "Conference",
                "Schulung": "Vocational Training",
                "Meeting": "Team Meeting",
                "Entwicklung": "Development",
            }
            df["TYPE_MISSION"] = df["TYPE_MISSION"].replace(map_type_mission)
        case "PARIS":
            map_type_mission = {
                "Conférence": "Conference",
                "Formation": "Vocational Training",
                "Réunion": "Team Meeting",
                "Rencontre entreprises": "Business Meeting",
                "Développement": "Development",
            }
            df["TYPE_MISSION"] = df["TYPE_MISSION"].replace(map_type_mission)

#### Conversion des fuseaux horaires

In [9]:
def clean_date(df, site, date_col="DATE_MISSION"):
    """
    Convertit la colonne de dates d'un site donné vers UTC+2 (heure de Paris).
    Les dates sont supposées être en heure locale du site (naive).
    """
    # Mapping site -> fuseau horaire IANA
    site_tz = {
        "PARIS":      "Europe/Paris",
        "BERLIN":     "Europe/Berlin",
        "LONDON":     "Europe/London",
        "NEWYORK":    "America/New_York",
        "LOSANGELES": "America/Los_Angeles",
        "SHANGHAI":   "Asia/Shanghai",
    }

    tz = site_tz[site]

    # 1) S'assurer que la colonne est bien en datetime
    df[date_col] = pd.to_datetime(df[date_col])

    # 2) Localiser la date dans le fuseau du site, puis convertir vers Paris (UTC+2 en été)
    df[date_col] = (
        df[date_col]
          .dt.tz_localize(tz)             # on dit "cette heure est en heure locale du site"
          .dt.tz_convert("Europe/Paris")  # on la convertit vers Paris
          .dt.tz_localize(None)           # on retire l'info de fuseau pour rester naive
    )

    return df

#### Chargement et union des données du personnel

In [10]:
def extract_personnel():
    sites = ["PARIS", "BERLIN", "LONDON", "NEWYORK", "SHANGHAI", "LOSANGELES"]
    base_path = Path("data/BDD_BGES")

    personnel_dfs = []

    # On charge la liste du personnel de chaque site dans un dataframe
    for site in sites:
        file_path = base_path / f"BDD_BGES_{site}" / f"PERSONNEL_{site}.txt"
        df = pd.read_csv(str(file_path), sep=';')
        clean_langue_fonction(df, site) 
        df['ID_SITE'] = site
        
        personnel_dfs.append(df)

    # On combine les dataframes 
    personnel_df = pd.concat(personnel_dfs)
  
    # On sélectionne uniquement les colonnes nécessaires
    personnel_final_df = personnel_df[['ID_PERSONNEL','FONCTION_PERSONNEL', 'ID_SITE']]
    return personnel_final_df

In [11]:
personnel_df = extract_personnel()
personnel_df.head()

,ID_PERSONNEL,FONCTION_PERSONNEL,ID_SITE
0,KeyPers_Paris_1230000,Business Executive,PARIS
1,KeyPers_Paris_1230001,HRD,PARIS
2,KeyPers_Paris_1230002,Data Engineer,PARIS
3,KeyPers_Paris_1230003,Computer Engineer,PARIS
4,KeyPers_Paris_1230004,Computer Engineer,PARIS


#### Traitement journalier des données de matériel informatique

In [ ]:
def extract_materiel(start_date, end_date):
    sites = ["PARIS", "BERLIN", "LONDON", "NEWYORK", "SHANGHAI", "LOSANGELES"]
    base_path = Path("data/BDD_BGES")
    # Liste pour stocker les dataframes de chaque jour
    all_materiel_dfs = []

    # Boucle sur chaque jour
    for single_date in pd.date_range(start_date, end_date):
        day_str = single_date.strftime("%Y%m%d")
        
        # Liste pour stocker les dataframes de matériel de la journée
        materiel_day_dfs = []
        
        # Boucle sur chaque site
        for site in sites:
            file_path = base_path / f"BDD_BGES_{site}/BDD_BGES_{site}_INFORMATIQUE/MATERIEL_INFORMATIQUE_{day_str}.txt"
            
            # Vérifier si le fichier existe avant de le lire
            if file_path.exists():
                df = pd.read_csv(str(file_path), sep=';')
                df['DATE'] = single_date #ToDo a voir si utile
                clean_date(df, site)
                materiel_day_dfs.append(df)
                
        # Si des fichiers ont été trouvés pour ce jour
        if materiel_day_dfs:
            # Combiner les données de tous les sites pour la journée et l'ajouter à la liste globale
            all_materiel_dfs.append(pd.concat(materiel_day_dfs, ignore_index=True))

    # Après la boucle, vérifier si on a collecté des données
    if all_materiel_dfs:
        # Combiner les données de tous les jours en un seul DataFrame
        materiel_total_df = pd.concat(all_materiel_dfs, ignore_index=True)
        return materiel_total_df
    else:
        print("Aucun fichier de matériel trouvé pour la période spécifiée.")

In [13]:
start_date = date(2026, 4, 29)
end_date = date(2026, 5, 25) 

mat_df = extract_materiel(start_date, end_date)
mat_df.head()

,ID_MATERIELINFO,ID_PERSONNEL,NOM_PERSONNEL,PRENOM_PERSONNEL,DATE_ACHAT,TYPE,MODELE,DATE
0,Paris_MATERIEL_INFO_202604290,KeyPers_Paris_1232362,Name2362,FistName2362,2026-04-29 08:17:31,PC fixe sans ecran,Z,2026-04-29
1,Paris_MATERIEL_INFO_202604291,KeyPers_Paris_1232165,Name2165,FistName2165,2026-04-29 09:42:55,imprimante,Laser A3 (>100kg),2026-04-29
2,Paris_MATERIEL_INFO_202604292,KeyPers_Paris_1231951,Name1951,FistName1951,2026-04-29 13:58:12,PC fixe sans ecran,Precision tower 3xxx,2026-04-29
3,Paris_MATERIEL_INFO_202604293,KeyPers_Paris_1230614,Name614,FistName614,2026-04-29 13:19:31,Telephone IP,,2026-04-29
4,Paris_MATERIEL_INFO_202604294,KeyPers_Paris_1232952,Name2952,FistName2952,2026-04-29 13:55:41,,modèle par défaut,2026-04-29


#### Traitement journalier des données de missions

In [14]:
# Même logique que la fonction d'extraction du matériel
def extract_missions(start_date, end_date):
    sites = ["PARIS", "BERLIN", "LONDON", "NEWYORK", "SHANGHAI", "LOSANGELES"]
    base_path = Path("data/BDD_BGES")
    
    all_missions_dfs = []

    for single_date in pd.date_range(start_date, end_date):
        day_str = single_date.strftime("%Y%m%d")
        
        missions_day_dfs = []
        
        for site in sites:
            file_path = base_path / f"BDD_BGES_{site}/BDD_BGES_{site}_MISSION/MISSION_{day_str}.txt"
            
            if file_path.exists():
                df = pd.read_csv(str(file_path), sep=';')
                clean_langue_mission(df, site)
                clean_date(df, site)
                df['ID_SITE'] = site
                missions_day_dfs.append(df)
                
        if missions_day_dfs:
            all_missions_dfs.append(pd.concat(missions_day_dfs, ignore_index=True))

    if all_missions_dfs :
        missions_total_df = pd.concat(all_missions_dfs, ignore_index=True)
        return missions_total_df
    else:
        print("Aucun fichier de missions trouvé pour la période spécifiée.")

In [15]:
start_date = date(2026, 4, 29)
end_date = date(2026, 5, 25) 

missions_df = extract_missions(start_date, end_date)
missions_df.head()

,ID_MISSION,ID_PERSONNEL,NOM_PERSONNEL,PRENOM_PERSONNEL,DATE_MISSION,TYPE_MISSION,VILLE_DEPART,PAYS_DEPART,VILLE_DESTINATION,PAYS_DESTINATION,TRANSPORT,ALLER_RETOUR,ID_SITE
0,Paris_202604290,KeyPers_Paris_1233578,Name3578,FistName3578,2026-04-29 15:01:12,Business Meeting,Paris,France,New-York,USA,Avion,oui,PARIS
1,Paris_202604291,KeyPers_Paris_1234968,Name4968,FistName4968,2026-04-29 18:41:44,Development,Paris,France,London,England,Avion,oui,PARIS
2,Paris_202604292,KeyPers_Paris_1233110,Name3110,FistName3110,2026-04-29 09:13:25,Business Meeting,Paris,France,Washington,USA,Avion,oui,PARIS
3,Paris_202604293,KeyPers_Paris_1230093,Name93,FistName93,2026-04-29 13:04:44,Conference,Paris,France,Berlin,Allemagne,Avion,oui,PARIS
4,Paris_202604294,KeyPers_Paris_1234582,Name4582,FistName4582,2026-04-29 15:34:56,Development,Paris,France,Montreal,Canada,Avion,oui,PARIS


#### Création tables de dimension

In [ ]:
sdf_materiel = spark.createDataFrame(mat_df)
sdf_personnel = spark.createDataFrame(personnel_df)
sdf_mission = spark.createDataFrame(missions_df)

data = [("BERLIN",), ("LONDON",), ("LOSANGELES",), ("NEWYORK",), ("PARIS",), ("SHANGHAI",)]
sdf_sites = spark.createDataFrame(data).toDF("ID_SITE")

In [ ]:
fait_materiel = (
    sdf_personnel
    .join(sdf_materiel, "ID_PERSONNEL", "inner")
)

In [31]:
fait_materiel.show()

+--------------------+------------------+--------+--------------------+-------------+----------------+-------------------+------------------+--------------------+-------------------+
|        ID_PERSONNEL|FONCTION_PERSONNEL| ID_SITE|     ID_MATERIELINFO|NOM_PERSONNEL|PRENOM_PERSONNEL|         DATE_ACHAT|              TYPE|              MODELE|               DATE|
+--------------------+------------------+--------+--------------------+-------------+----------------+-------------------+------------------+--------------------+-------------------+
|KeyPers_Paris_123...| Computer Engineer|   Paris|Paris_MATERIEL_IN...|     Name1848|    FistName1848|2026-05-02 16:45:25|       PC portable|       Latitude 3xxx|2026-05-02 00:00:00|
|KeyPers_Berlin_12...| Computer Engineer|  Berlin|BERLIN_MATERIEL_I...|     Name2888|    FistName2888|2026-04-29 16:30:47|PC fixe sans ecran|                    |2026-04-29 00:00:00|
|KeyPers_NewYork_1...|     Data Engineer| NewYork|NewYork_MATERIEL_...|     Name2864|